### Lesson 1.2 - Statistical Models (BoW, TF-IDF)
---

<b>1.2.1: Why Statistical Models?</b><br>
Using BPE we break the text into token, still they are in the form of strings. Neural network is trained on numbers not text.

Question: How can we convert a sentence into a vector?
-> By using statistical models

<b>1.2.2: Bag of Words (BoW) - The Simplest Approach</b><br>
Concept:
- Which words are in a sentence, count them
- We ignore the ordering of the words(that's why we call it "Bag")
- Every word becomes a dimension

Example:
```js
Sentence 1: "the cat sat"
Sentence 2: "the dog sat"
Sentence 3: "the cat and dog played"

Vocabulary: {'the': 0, 'cat': 1, 'sat': 2, 'dog': 3, 'and': 4, 'played': 5}

BoW Vectors:
Sentence 1: [1, 1, 1, 0, 0, 0]  (the=1, cat=1, sat=1, baaki 0)
Sentence 2: [1, 0, 1, 1, 0, 0]  (the=1, dog=1, sat=1)
Sentence 3: [1, 1, 0, 1, 1, 1]  (the=1, cat=1, dog=1, and=1, played=1)
```

Problem with BoW:
- Words like "the" are in most of the sentences, it's importance should be less
- Words like "machine" are rare in a document which are more meaningful
- BoW gives both of them the same weight - this is wrong

<b>1.2.3: TF-IDF - The Smart Approach</b><br>
Concept:
- TF - How many times a single word appears in a sentence
- IDF - A single word appears in how many sentences
- TF-IDF = TF * IDF

Intuition:
- If a word appears most of the times in a sentence (high TF) and very less accross all of the sentences then (high IDF) then it's importance is HIGH.
- If a word appears in every sentence (low IDF), thus it has a lower TF-IDF score.

### Math behind it:
---

Step 1: Term Frequency (TF)

```js
TF(word, sentence) = (word count in sentence)/(total word in sentence)
```
Example:
```js
Sentence: "the cat sat the cat"
Total words: 5
TF("the") = 2/5 = 0.4
TF("cat") = 2/5 = 0.4
TF("sat") = 1/5 = 0.2
```

Step 2: Inverse Document Frequence (IDF)

```js
IDF(word) = log(Total sentences)/(Sentences containing word)
```
Example:
```js
Total sentences: 3
"the" appears in: 3 sentences → IDF("the") = log(3/3) = log(1) = 0
"cat" appears in: 2 sentences → IDF("cat") = log(3/2) = 0.405
"sat" appears in: 2 sentences → IDF("sat") = log(3/2) = 0.405
"dog" appears in: 2 sentences → IDF("dog") = log(3/2) = 0.405
"and" appears in: 1 sentence → IDF("and") = log(3/1) = 1.099
"played" appears in: 1 sentence → IDF("played") = log(3/1) = 1.099
```

Step 3: TF-IDF Score

```js
TF-IDF(word, sentence) = TF(word, sentence) x IDF(word)
```
Example:
```js
Sentence 1: "the cat sat"
TF-IDF("the") = (1/3) × 0 = 0  ← Noise eliminated!
TF-IDF("cat") = (1/3) × 0.405 = 0.135
TF-IDF("sat") = (1/3) × 0.405 = 0.135
```

Step 4: Cosine Similarity

```js
Cosine Similarity(A, B) = (A · B) / (||A|| × ||B||)
```
Example:
```js
Sentence 1: [0, 0.135, 0.135, 0, 0, 0]
Sentence 2: [0, 0, 0.135, 0.135, 0, 0]

Dot product: 0×0 + 0.135×0 + 0.135×0.135 + 0×0.135 + 0×0 + 0×0 = 0.018
||A|| = sqrt(0² + 0.135² + 0.135²) = 0.191
||B|| = sqrt(0² + 0² + 0.135² + 0.135²) = 0.191

Cosine Similarity = 0.018 / (0.191 × 0.191) = 0.494
```

In [ ]:
# Vocab Building

corpus = [
    "the cat sat",
    "the dog sat",
    "the cat and dog played"
]

vocab = {}
count = 0

for sent in corpus:
    sent = sent.split()
    for word in sent:
        if word not in vocab:
            vocab[word] = count
            count += 1

print(f"Vocab: {vocab}")
print(f"Vocab size: {len(vocab)}")

Vocab: {'the': 0, 'cat': 1, 'sat': 2, 'dog': 3, 'and': 4, 'played': 5}
Vocab size: 6


In [8]:
# TF matrix

import torch

num_docs = len(corpus)
vocab_size = len(vocab)

tf_matrix = torch.zeros((num_docs, vocab_size), dtype=torch.long)

for i, sent in enumerate(corpus):
    words = sent.split()
    for word in words:
        j = vocab[word]
        tf_matrix[i, j] += 1

print(f"Raw TF Matrix:\n{tf_matrix}")

row_sums = tf_matrix.sum(dim=1, keepdim=True)
tf_matrix_normalized = tf_matrix/row_sums
tf_matrix_normalized

Raw TF Matrix:
tensor([[1, 1, 1, 0, 0, 0],
        [1, 0, 1, 1, 0, 0],
        [1, 1, 0, 1, 1, 1]])


tensor([[0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.0000, 0.3333, 0.3333, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.0000, 0.2000, 0.2000, 0.2000]])

In [15]:
# IDF implementation

df_vector = tf_matrix.sum(dim=0).float()
N = torch.tensor(num_docs, dtype=torch.long)

idf = torch.log(N / df_vector)
tf_idf_matrix = tf_matrix_normalized * idf

tf_idf_matrix

tensor([[0.0000, 0.1352, 0.1352, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.1352, 0.1352, 0.0000, 0.0000],
        [0.0000, 0.0811, 0.0000, 0.0811, 0.2197, 0.2197]])

In [19]:
# Cosine similarity

def cosine_sim(vec1, vec2):
    numerator = torch.dot(vec1, vec2)
    denominator = torch.norm(vec1) * torch.norm(vec2)
    sim_score = numerator/denominator
    return sim_score

for i in range(len(tf_idf_matrix) - 1):
    doc1 = tf_idf_matrix[i]
    doc2 = tf_idf_matrix[i+1]
    sim_score = cosine_sim(doc1, doc2)
    print(f"Similarity score: {sim_score:.4f}")

Similarity score: 0.5000
Similarity score: 0.1731
